# Sheet 5 solutions: Iterative methods
Run the notebook from top to bottom. The code uses NumPy, SciPy and Matplotlib; no data files are needed. Each subquestion has its own explanation and short code blocks. Plots use ordinary lines, markers, labels and legends.

Numbers printed below are numerical approximations. The PDF contains the results of one complete execution; function-evaluation counts and the final printed digits may differ slightly between environments.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import root, least_squares


## Question 1: Rates of convergence
### 1(a) Compute guesses and errors
For these examples the roots are known, so we can measure the actual error $e_k=|x_k-x^*|$. In general, $|f(x_k)|$ is not this error.

The derivatives are
$$
f_1'(x)=2x,\qquad f_2'(x)=2(x-1),\qquad f_3'(x)=1+3x^2.
$$
First define each function and its derivative. The short loop below uses the known root only to stop this experiment; it is not a general root-finding stopping rule.

In [ ]:
def f1(x):
    return x*x - 2.0

def df1(x):
    return 2.0*x

def f2(x):
    return (x - 1.0)**2

def df2(x):
    return 2.0*(x - 1.0)

def f3(x):
    return x + x**3

def df3(x):
    return 1.0 + 3.0*x*x


In [ ]:
# Store each sequence separately for later plots and fits.
cases = [("f1", f1, df1, 2.0, np.sqrt(2.0)),
         ("f2", f2, df2, 2.0, 1.0),
         ("f3", f3, df3, 0.5, 0.0)]
rate_errors = {}
rate_guesses = {}

for name, f, df, initial, exact_root in cases:
    guesses = [initial]
    x = initial
    for k in range(12):
        if abs(x - exact_root) < 1e-12:
            break
        x = x - f(x)/df(x)
        guesses.append(x)
    guesses = np.array(guesses)
    errors = np.abs(guesses - exact_root)
    rate_guesses[name] = guesses
    rate_errors[name] = errors
    print(name)
    print("Guesses:", guesses)
    print("Errors: ", errors)


In [ ]:
# A logarithmic vertical axis keeps very small errors visible.
plt.figure()
for name in rate_errors:
    errors = rate_errors[name]
    iterations = np.arange(len(errors))
    positive = errors > 0.0
    plt.semilogy(iterations[positive], errors[positive], "o-", label=name)
plt.xlabel("Iteration k")
plt.ylabel("True error |x_k - x*| (log scale)")
plt.legend()
plt.show()


The linear sequence decreases by a roughly fixed factor at each update; the faster sequences bend down more sharply. `semilogy` changes only the vertical scale. Equal vertical distances represent equal multiplicative changes, not equal amounts subtracted. Zero errors cannot be shown on a logarithmic axis.

Newton need not have exactly order two. For the repeated root of $f_2$, simplifying the update gives
$$
x_{k+1}=\frac{x_k+1}{2},\qquad e_{k+1}=\frac12 e_k.
$$
Thus its order is one, with $C=1/2$. For $f_1$ near the positive root, $e_{k+1}=e_k^2/(2x_k)$, so the order is two and $C\to1/(2\sqrt2)$. For $f_3$,
$$
x_{k+1}=\frac{2x_k^3}{1+3x_k^2},\qquad
 e_{k+1}=\frac{2e_k^3}{1+3e_k^2}.
$$
The special form of this function gives order three and $C\to2$. A simple root usually gives quadratic Newton convergence under the usual smoothness assumptions, but higher orders can occur in special cases.

### 1(b) Estimate the order by two regressions
Write $u_k=e_k$ and $v_k=e_{k+1}$. Fit the model $v=C u^p$ to the positive pairs, excluding zeros caused by rounding.

In the log fit, $\log v=\beta_0+p\log u$: the slope is $p$, and $C=\exp(\beta_0)$. The two columns of the design matrix are a constant and $\log u$. In the direct fit, `least_squares` changes $[C,p]$ to make the vector $C u^p-v$ small.

In [ ]:
def direct_rate_differences(parameters, previous, following):
    C, p = parameters
    differences = C*previous**p - following
    # One common scale prevents tiny values causing an early numerical stop.
    # It does not change which parameters minimise the squared differences.
    return differences / np.max(following)

def fit_error_pairs(previous, following):
    A = np.column_stack([np.ones(len(previous)), np.log(previous)])
    intercept, p = np.linalg.lstsq(A, np.log(following), rcond=None)[0]
    log_parameters = np.array([np.exp(intercept), p])
    direct_fit = least_squares(
        direct_rate_differences, [1.0, 2.0],
        args=(previous, following),
        bounds=([1e-8, 1e-8], [np.inf, np.inf]),
        ftol=1e-12, xtol=1e-12, gtol=1e-12, max_nfev=1000)
    return log_parameters, direct_fit


The common scale in the direct residual vector changes its numerical size, but not its minimiser: it multiplies the whole squared objective by one positive constant. We still compare the original, unscaled differences below. Tighter solver tolerances help when there are only a few very small error pairs; these tolerances are not error bounds on $C$ or $p$.

In [ ]:
def rate_scores(parameters, previous, following):
    C, p = parameters
    raw = C*previous**p - following
    logged = np.log(C) + p*np.log(previous) - np.log(following)
    return raw, np.sum(raw**2), np.sum(logged**2)

rate_fits = {}
for name in rate_errors:
    errors = rate_errors[name]
    previous, following = errors[:-1], errors[1:]
    keep = (previous > 0.0) & (following > 0.0)
    previous, following = previous[keep], following[keep]
    log_parameters, direct_fit = fit_error_pairs(previous, following)
    rate_fits[name] = (previous, following, log_parameters, direct_fit)
    print(name, "direct solver success:", direct_fit.success)
    for method, parameters in [("Log fit", log_parameters),
                               ("Direct fit", direct_fit.x)]:
        raw, raw_sse, log_sse = rate_scores(parameters, previous, following)
        print(method, "[C, p] =", parameters)
        print("Original-scale SSE:", raw_sse, "Log-scale SSE:", log_sse)


In [ ]:
# Compare both methods on the same original error scale.
for name in rate_fits:
    previous, following, log_parameters, direct_fit = rate_fits[name]
    log_raw = rate_scores(log_parameters, previous, following)[0]
    direct_raw = rate_scores(direct_fit.x, previous, following)[0]
    pair_index = np.arange(len(previous))
    plt.figure()
    plt.plot(pair_index, log_raw, "o-", label="Log fit")
    plt.plot(pair_index, direct_raw, "s--", label="Direct fit")
    plt.axhline(0.0, color="black", linestyle=":")
    plt.xlabel("Error-pair index k")
    plt.ylabel("C e_k**p - e_(k+1)")
    plt.title(name + ": differences on the original scale")
    plt.legend()
    plt.show()


In [ ]:
# Remove the first pair and repeat exactly the same two fits.
local_rate_fits = {}
for name in rate_fits:
    previous, following = rate_fits[name][:2]
    log_parameters, direct_fit = fit_error_pairs(previous[1:], following[1:])
    local_rate_fits[name] = (log_parameters, direct_fit)
    print(name, "without first pair")
    print("Log [C, p]:", log_parameters)
    print("Direct [C, p]:", direct_fit.x)


The later-pair fits suggest orders near two, one and three. The first Newton updates are not necessarily in the asymptotic regime, so fitting all pairs need not return the exact theoretical order. Removing early pairs often moves the estimate towards the local rate, but also leaves fewer observations.

The direct fit minimises squared differences on the original scale and therefore gives more weight to larger absolute errors. The log fit minimises squared log differences; proportionate mismatches at small errors matter more. Each fit should perform best, or nearly best within numerical accuracy, on its own objective. For $f_2$ the power law is exact, so both fits agree closely. Its plotted differences are around $10^{-16}$: the automatic vertical scale magnifies rounding effects that are numerically negligible. These artificial error pairs are not independent noisy observations; regression here is a descriptive rate estimate, not a statistical confidence statement.

### 1(c) Starting guesses and local behaviour
A practical Newton loop cannot normally use the unknown true root as its stopping test. This reusable version checks the function value, limits the number of updates, and reports a zero derivative or a non-finite calculation. It checks the initial point before attempting an update.

In [ ]:
def newton_history(f, df, initial, tol=1e-10, max_steps=50):
    x = float(initial)
    history = [x]
    for k in range(max_steps):
        value = f(x)
        if not np.isfinite(value):
            return np.array(history), "Non-finite function value"
        if abs(value) < tol:
            return np.array(history), "Function-value tolerance met"
        derivative = df(x)
        if derivative == 0.0:
            return np.array(history), "Zero derivative"
        if not np.isfinite(derivative):
            return np.array(history), "Non-finite derivative"
        next_x = x - value/derivative
        if not np.isfinite(next_x):
            return np.array(history), "Non-finite next guess"
        x = next_x
        history.append(x)
    if abs(f(x)) < tol:
        return np.array(history), "Function-value tolerance met"
    return np.array(history), "Iteration limit reached"


In [ ]:
starting_results = {}
for initial in [0.1, 10.0, -2.0, 0.0]:
    history, message = newton_history(f1, df1, initial, max_steps=30)
    starting_results[initial] = (history, message)
    print("Start:", initial, "Final:", history[-1])
    print("|f(x)|:", abs(f1(history[-1])), "Updates:", len(history)-1)
    print(message)


The positive starting guesses reach $+\sqrt2$; starting at $-2$ reaches $-\sqrt2$. Starting at zero makes the derivative zero, so the basic step is undefined. A fast local rate describes what happens after getting close to a suitable root. It does not promise convergence to a chosen root from every starting point. In addition, a small function value must be interpreted using the equation's scale.

## Question 2: Newton and root
### 2(a) The equation and its graph
$$
f(x)=e^x+x^3-5,\qquad f'(x)=e^x+3x^2>0.
$$
The function is strictly increasing, so it has at most one real root. Since $f(0)=-4$ and $f(2)=e^2+3>0$, continuity gives exactly one root, between 0 and 2.

In [ ]:
def complicated_f(x):
    return np.exp(x) + x**3 - 5.0

def complicated_df(x):
    return np.exp(x) + 3.0*x*x

plot_x = np.linspace(-2.0, 2.0, 300)
plt.figure()
plt.plot(plot_x, complicated_f(plot_x), label="f(x)")
plt.axhline(0.0, color="black", linestyle="--")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.legend()
plt.show()


### 2(b) Manual Newton from zero
Reuse the Newton function from 1(c). The update count is the number of stored points minus one, because the first point is the initial guess.

In [ ]:
manual_scalar, manual_message = newton_history(
    complicated_f, complicated_df, 0.0)
print("Estimate:", manual_scalar[-1])
print("Absolute function value:", abs(complicated_f(manual_scalar[-1])))
print("Updates:", len(manual_scalar)-1)
print("Tolerance met:", abs(complicated_f(manual_scalar[-1])) < 1e-10)
print(manual_message)


### 2(c) SciPy root from zero
`root` expects a vector of equation values. Here there is one unknown and one equation, so the input and output each have one entry. `sol.x` is the returned point, `sol.fun` contains the equation value there, and `success` and `message` describe solver termination. We independently check the function-value target, because the hybrid method's tolerance is not a bound on this value or on the distance to the true root.

In [ ]:
def scalar_equation(z):
    return np.array([complicated_f(z[0])])

scalar_sol = root(scalar_equation, [0.0], method="hybr", tol=1e-10)
print("Success:", scalar_sol.success)
print("Message:", scalar_sol.message)
print("Returned point:", scalar_sol.x)
print("Function value:", scalar_sol.fun)
print("Function-value check:", abs(scalar_sol.fun[0]) < 1e-10)


### 2(d) Compare starting guesses
Use the same initial guess for the two methods in each comparison. The hybrid method controls its steps and may estimate derivatives; it need not follow the manual Newton sequence. Its function-evaluation count is not an iteration count and does not include every kind of computational work.

In [ ]:
scalar_comparisons = {}
plt.figure()
for initial in [0.0, -1.0, 4.0]:
    history, message = newton_history(complicated_f, complicated_df, initial)
    sol = root(scalar_equation, [initial], method="hybr", tol=1e-10)
    scalar_comparisons[initial] = (history, sol)
    print("Start:", initial)
    print("Newton:", history[-1], "updates:", len(history)-1, message)
    print("root:", sol.x[0], "success:", sol.success, "nfev:", sol.nfev)
    print("|f| for Newton and root:", abs(complicated_f(history[-1])),
          abs(sol.fun[0]))
    values = np.abs(complicated_f(history))
    plt.plot(np.arange(len(history)), values, "o-", label=str(initial))
plt.xlabel("Newton update k")
plt.ylabel("Absolute function value")
plt.legend(title="Starting guess")
plt.show()


All three starts lead to approximately 1.193674575 in this experiment. The first Newton step from zero jumps to 4 and initially increases the function magnitude; subsequent steps recover. This shows why initial behaviour can differ from the eventual local rate. The ordinary vertical axis makes that initial jump visible, although later tiny values are harder to distinguish.

## Question 3: Bracketing a root
### 3(a) Newton can cycle
For $f(x)=x^3-2x+2$, the derivative is $3x^2-2$. From zero, the Newton sequence alternates between 0 and 1. Neither point is a root. This is a failure of that starting guess, not evidence that the equation has no root.

In [ ]:
def bracket_f(x):
    return x**3 - 2.0*x + 2.0

def bracket_df(x):
    return 3.0*x*x - 2.0

plot_x = np.linspace(-2.0, 2.0, 300)
plt.figure()
plt.plot(plot_x, bracket_f(plot_x))
plt.axhline(0.0, color="black", linestyle="--")
plt.xlabel("x")
plt.ylabel("x**3 - 2x + 2")
plt.show()

for initial in [0.0, -2.0]:
    x = initial
    guesses = [x]
    for k in range(6):
        x = x - bracket_f(x)/bracket_df(x)
        guesses.append(x)
    print("Start:", initial)
    print("Guesses:", np.array(guesses))
    print("Function values:", bracket_f(np.array(guesses)))


### 3(b) Keep an interval containing a root
Here $f(-2)=-2$ and $f(-1)=3$. At each midpoint, retain the half whose endpoints have opposite function signs. Continuity guarantees that a root remains inside. We stop using bracket width in this part, not the function value. `brackets[0]` is the original interval, so its index also counts halvings.

In [ ]:
a, b = -2.0, -1.0
brackets = [(a, b)]
for k in range(50):
    if b-a < 2e-8:
        break
    midpoint = (a+b)/2.0
    if bracket_f(midpoint) == 0.0:
        a, b = midpoint, midpoint
    elif bracket_f(a)*bracket_f(midpoint) < 0.0:
        b = midpoint
    else:
        a = midpoint
    brackets.append((a, b))

brackets = np.array(brackets)
bisection_estimate = (a+b)/2.0
print("Estimate:", bisection_estimate)
print("Absolute function value:", abs(bracket_f(bisection_estimate)))
print("Final bracket:", a, b)
print("Bracket width:", b-a)
print("Halvings:", len(brackets)-1)


### 3(c) A guaranteed error bound
The midpoint is no farther than half the interval width from any root in that interval. Every halving divides the width by two, giving
$$
|m_k-x^*|\le\frac{b_k-a_k}{2}
=\frac{1}{2^{k+1}}.
$$
To guarantee a strict error below $10^{-8}$, we need $2^{-(k+1)}<10^{-8}$, or $k>\log_2(10^8)-1\approx25.575$. Therefore 26 halvings suffice. This bound halves at each step, even though the actual midpoint error need not decrease at every step. If an exact midpoint root is found early, the interval collapses and the geometric expression remains an upper bound.

In [ ]:
widths = brackets[:, 1] - brackets[:, 0]
error_bounds = widths/2.0
print("Final guaranteed error bound:", error_bounds[-1])
plt.figure()
plt.semilogy(np.arange(len(error_bounds)), error_bounds, "o-")
plt.xlabel("Number of halvings k")
plt.ylabel("Midpoint error bound (log scale)")
plt.show()


### 3(d) Compare using the same function-value target
We repeat bisection with the stopping condition used for Newton in this comparison. This is a different stopping test from 3(b), so its update count will differ. Count bisection updates as interval halvings and Newton updates as corrections. The function-value targets agree, but the internal work of these updates differs.

In [ ]:
a, b = -2.0, -1.0
halvings = 0
for k in range(50):
    midpoint = (a+b)/2.0
    if abs(bracket_f(midpoint)) < 1e-10:
        break
    if bracket_f(a)*bracket_f(midpoint) < 0.0:
        b = midpoint
    else:
        a = midpoint
    halvings += 1
midpoint = (a+b)/2.0
newton_bracket, newton_bracket_message = newton_history(
    bracket_f, bracket_df, -2.0)
print("Newton:", newton_bracket[-1], "updates:", len(newton_bracket)-1)
print("Newton |f|:", abs(bracket_f(newton_bracket[-1])))
print(newton_bracket_message)
print("Bisection:", midpoint, "halvings:", halvings)
print("Bisection |f|:", abs(bracket_f(midpoint)))
print("Final bracket width:", b-a)


Newton is faster from this suitable starting guess, but needs derivatives and can cycle from zero. Bisection needs a continuous function and an initial sign-changing bracket; it needs no derivative and maintains a bound on the root's location. These are different strengths. A fast local rate alone does not establish which method is appropriate for every problem.

## Question 4: Gauss-Newton and root
### 4(a) Equations, Jacobian and curves
$$
F(u,v)=\begin{bmatrix}u^2+v-3\\u+v^2-3\end{bmatrix},\qquad
J(u,v)=\begin{bmatrix}2u&1\\1&2v\end{bmatrix}.
$$
Each Jacobian row differentiates one equation; its columns correspond to $u$ and $v$. The graph can be drawn without a mesh: parameterise the first curve by $u$, and the second by $v$.

In [ ]:
def F(z):
    u, v = z
    return np.array([u*u + v - 3.0, u + v*v - 3.0])

def J(z):
    u, v = z
    return np.array([[2.0*u, 1.0], [1.0, 2.0*v]])

def draw_equation_curves():
    coordinates = np.linspace(-3.0, 3.0, 400)
    plt.plot(coordinates, 3.0-coordinates**2, label="u squared + v = 3")
    plt.plot(3.0-coordinates**2, coordinates, label="u + v squared = 3")
    plt.xlim(-3.0, 3.0)
    plt.ylim(-3.0, 3.0)
    plt.xlabel("u")
    plt.ylabel("v")

print("Shapes:", F([1.0, 1.0]).shape, J([1.0, 1.0]).shape)
plt.figure()
draw_equation_curves()
plt.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
plt.show()


The legend is placed beside the graph so it does not cover an intersection. There are four intersections. Subtracting the two equations gives $(u-v)(u+v-1)=0$. The branch $u=v$ gives $u=v=(-1\pm\sqrt{13})/2$; the branch $u+v=1$ gives $(2,-1)$ and $(-1,2)$.

### 4(b) A single correction
Near $z$, make the linear approximation $F(z+\Delta z)\approx F(z)+J(z)\Delta z$. The least-squares call returns the correction as the first item of its result. Add it to $z$ to obtain the next point.

In [ ]:
def gauss_newton_step(F, J, z):
    correction = np.linalg.lstsq(J(z), -F(z), rcond=None)[0]
    return z + correction

first_point = gauss_newton_step(F, J, np.array([1.0, 1.0]))
print("First updated point:", first_point)
print("Function values there:", F(first_point))


If the Jacobian is square and nonsingular, the linear system $J\Delta z=-F$ has one solution. That solution makes the linear least-squares objective exactly zero, so this Gauss-Newton step equals the ordinary Newton step. The original nonlinear equations need not be zero after one step: the approximation has omitted nonlinear terms.

### 4(c) Repeat the corrections
Check the initial point first, then perform up to 30 corrections. The Euclidean norm of the two equation values is a single measure of how nearly both equations are satisfied.

In [ ]:
def gauss_newton_history(F, J, initial, tol=1e-10, max_steps=30):
    z = np.array(initial, dtype=float)
    history = [z.copy()]
    for k in range(max_steps):
        if np.linalg.norm(F(z)) < tol:
            break
        z = gauss_newton_step(F, J, z)
        history.append(z.copy())
    return np.array(history)

system_history = gauss_newton_history(F, J, [1.0, 1.0])
print("Final point:", system_history[-1])
print("Function-value norm:", np.linalg.norm(F(system_history[-1])))
print("Updates:", len(system_history)-1)
print("Tolerance met:", np.linalg.norm(F(system_history[-1])) < 1e-10)


### 4(d) Compare root and another initial guess
Pass the same functions F and J to the hybrid solver. Collect the two manual histories, and mark both manual and library results on the curve plot. Different marker shapes distinguish the results when their positions agree.

In [ ]:
system_comparisons = []
for initial in [[1.0, 1.0], [2.0, 0.0]]:
    history = gauss_newton_history(F, J, initial)
    sol = root(F, initial, jac=J, method="hybr", tol=1e-10)
    system_comparisons.append((initial, history, sol))
    print("Start:", initial)
    print("Manual:", history[-1], "norm:", np.linalg.norm(F(history[-1])))
    print("root:", sol.x, "norm:", np.linalg.norm(sol.fun))
    print("Success:", sol.success, "Message:", sol.message)

plt.figure()
draw_equation_curves()
for initial, history, sol in system_comparisons:
    plt.plot(history[-1, 0], history[-1, 1], "o", label="Manual " + str(initial))
    plt.plot(sol.x[0], sol.x[1], "x", label="root " + str(initial))
plt.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
plt.show()


In [ ]:
plt.figure()
for initial, history, sol in system_comparisons:
    values = np.array([np.linalg.norm(F(z)) for z in history])
    positive = values > 0.0
    plt.semilogy(np.arange(len(values))[positive], values[positive],
                 "o-", label=str(initial))
plt.xlabel("Manual Gauss-Newton update k")
plt.ylabel("Norm of equation values (log scale)")
plt.legend(title="Starting point")
plt.show()


Starting from (1,1) reaches approximately (1.302775638, 1.302775638). Starting from (2,0) reaches (2,-1). The methods agree on these examples, but neither call is a search for all intersections. The equation-value histories can initially grow; a basic full-step method does not guarantee a decrease at every update.

## Question 5: Nonlinear regression
### 5(a) Evaluate trial models
For each parameter pair, the model returns a full vector of predictions at the requested inputs. Changing a or b changes that curve. The parameter a controls the limiting height, while b determines the input at half that height: a larger b makes the rise slower for fixed a.

In [ ]:
inputs = np.array([0.0, 0.5, 1.0, 2.0, 3.0, 5.0, 8.0, 12.0])
observed = np.array([0.08, 1.92, 3.45, 4.94, 6.12, 7.05, 8.13, 8.49])

def model(parameters, inputs):
    a, b = parameters
    return a*inputs/(b + inputs)

trial_parameters = [[10.0, 2.0], [6.0, 2.0], [10.0, 5.0]]
plot_inputs = np.linspace(0.0, 15.0, 200)
plt.figure()
plt.plot(inputs, observed, "o", label="Measurements")
for parameters in trial_parameters:
    plt.plot(plot_inputs, model(parameters, plot_inputs), label=str(parameters))
plt.xlabel("Input concentration x")
plt.ylabel("Sensor response y")
plt.legend()
plt.show()


### 5(b) Residuals and automatic fitting
A residual here is a model prediction minus its corresponding observation. The residual function receives a trial parameter pair and returns eight differences. `least_squares` forms the sum of squares internally and proposes other parameter pairs. A prediction vector must therefore be evaluated inside the residual function, using the supplied pair.

In [ ]:
def residuals(parameters, inputs, observed):
    predictions = model(parameters, inputs)
    return predictions - observed

for parameters in trial_parameters:
    differences = residuals(parameters, inputs, observed)
    print(parameters, "sum of squares:", np.sum(differences**2))
print("Residual shape:", residuals([10.0, 2.0], inputs, observed).shape)

# Try a small manual improvement before using the optimiser.
manual_trial = [10.0, 1.98]
print("Manual trial:", manual_trial)
print("Sum of squares:", np.sum(residuals(manual_trial, inputs, observed)**2))


In [ ]:
sensor_fit = least_squares(
    residuals, [6.0, 4.0], args=(inputs, observed),
    bounds=([0.01, 0.01], [np.inf, np.inf]))
print("Fitted [a, b]:", sensor_fit.x)
print("Success:", sensor_fit.success)
print("Message:", sensor_fit.message)
print("Sum of squares:", np.sum(sensor_fit.fun**2))


The first trial pair is already close to the data, and a small change can improve it. The solver improves the pair further. The nonzero measurement at input zero cannot be fitted by this model: its prediction there is zero for every parameter choice, contributing a fixed squared difference of 0.0064. A nonzero final sum of squares is therefore expected.

### 5(c) Inspect the fit and change the starting guess
The fitted curve shows the predicted responses; the signed differences show which predictions lie above or below the data. Plot these separately because they have different meanings and scales. A horizontal zero line makes the signs easy to see.

In [ ]:
plt.figure()
plt.plot(inputs, observed, "o", label="Measurements")
plt.plot(plot_inputs, model(sensor_fit.x, plot_inputs), label="Fitted model")
plt.xlabel("Input concentration x")
plt.ylabel("Sensor response y")
plt.legend()
plt.show()

plt.figure()
plt.plot(inputs, sensor_fit.fun, "o-")
plt.axhline(0.0, color="black", linestyle="--")
plt.xlabel("Input concentration x")
plt.ylabel("Fitted response - observation")
plt.show()


In [ ]:
second_sensor_fit = least_squares(
    residuals, [15.0, 1.0], args=(inputs, observed),
    bounds=([0.01, 0.01], [np.inf, np.inf]))
print("Second fitted [a, b]:", second_sensor_fit.x)
print("Success:", second_sensor_fit.success)
print("Message:", second_sensor_fit.message)
print("Sum of squares:", np.sum(second_sensor_fit.fun**2))
print("Coefficient differences:", second_sensor_fit.x - sensor_fit.x)


The fitted differences have both signs, and the two starting guesses give very similar coefficients and squared errors here. That is reassuring for this experiment, but nonlinear least squares does not in general guarantee a unique or global best fit.

### 5(d) Predict without fitting again
Use the fitted coefficients as they are. Inputs 1.5, 4 and 10 lie inside the observed range [0,12]; 20 is outside it. This last prediction is extrapolation and depends on the chosen model continuing to apply beyond the measured range.

In [ ]:
new_inputs = np.array([1.5, 4.0, 10.0, 20.0])
new_predictions = model(sensor_fit.x, new_inputs)
for x, prediction in zip(new_inputs, new_predictions):
    print("Input:", x, "Prediction:", prediction)

extended_inputs = np.linspace(0.0, 20.0, 250)
plt.figure()
plt.plot(inputs, observed, "o", label="Measurements")
plt.plot(extended_inputs, model(sensor_fit.x, extended_inputs), label="Fitted model")
plt.plot(new_inputs, new_predictions, "x", label="New predictions")
plt.xlabel("Input concentration x")
plt.ylabel("Sensor response y")
plt.legend()
plt.show()
